In [1]:
import os, subprocess

REPO_URL = "https://github.com/tongyuguo/HelpHerInvest.git"
REPO_DIR = "HelpHerInvest"
data_dir = os.path.join(REPO_DIR, "Data")

def clone_or_pull():
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "pull"])
    else:
        subprocess.run(["git", "clone", REPO_URL])

clone_or_pull()

Already up to date.


In [2]:
'''
%pip install openai python-dotenv
%pip install -qU langchain-openai langchain
%pip install langchain_community pyowm
%pip install --upgrade --quiet  youtube_search
'''

'\n%pip install openai python-dotenv\n%pip install -qU langchain-openai langchain\n%pip install langchain_community pyowm\n%pip install --upgrade --quiet  youtube_search\n'

In [3]:
print(os.getcwd())

/home/jupyter-guotq


In [4]:
from pprint import pprint

#API 
from dotenv import load_dotenv
import os
load_dotenv() 


True

In [5]:
#testing if the API key is working

from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4",
    input="tell me about boston college"
)

print(response.output_text)

Boston College is a **private Jesuit research university** located in **Chestnut Hill, Massachusetts**, just outside Boston. It was founded in **1863** and is one of the better-known Catholic universities in the United States.

### Quick overview
- **Type:** Private, Jesuit, Catholic university  
- **Location:** Chestnut Hill, Massachusetts  
- **Founded:** 1863  
- **Nickname:** Eagles  
- **Colors:** Maroon and gold  

### Academics
Boston College is known for strong programs in:
- **Business** — especially through the **Carroll School of Management**
- **Liberal arts and sciences**
- **Education**
- **Nursing**
- **Law**
- **Theology and philosophy**

It has a reputation for combining rigorous academics with a Jesuit emphasis on ethics, service, and educating the “whole person.”

### Campus and culture
BC’s campus is famous for its **Gothic-style architecture**, especially on the original Chestnut Hill campus. The school has a traditional college feel, strong school spirit, and a co

In [6]:
'''
from openai import OpenAI
client = OpenAI()

file = client.files.create(
    file=open("./HelpHerInvest/Data/nlp_clean_stock_symbols.csv", "rb"),
    purpose="user_data"
)

response = client.responses.create(
    model="gpt-5-mini",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "file_id": file.id,
                },
                {
                    "type": "input_text",
                    "text": "Role: you are a data scientist in charge of filtering out stock symbels that is relevant to user input. Input: I like dessert food. Steps: you first analyze user input, then select relevant symbbols from file located in https://github.com/tongyuguo/HelpHerInvest/blob/main/Data/nlp_clean_stock_symbols.csv. Expectation: you are expected to return a list of 20 relebant stock symbosl given user imput.",
                },
            ]
        }
    ]
)

print(response.output_text)
'''

'\nfrom openai import OpenAI\nclient = OpenAI()\n\nfile = client.files.create(\n    file=open("./HelpHerInvest/Data/nlp_clean_stock_symbols.csv", "rb"),\n    purpose="user_data"\n)\n\nresponse = client.responses.create(\n    model="gpt-5-mini",\n    input=[\n        {\n            "role": "user",\n            "content": [\n                {\n                    "type": "input_file",\n                    "file_id": file.id,\n                },\n                {\n                    "type": "input_text",\n                    "text": "Role: you are a data scientist in charge of filtering out stock symbels that is relevant to user input. Input: I like dessert food. Steps: you first analyze user input, then select relevant symbbols from file located in https://github.com/tongyuguo/HelpHerInvest/blob/main/Data/nlp_clean_stock_symbols.csv. Expectation: you are expected to return a list of 20 relebant stock symbosl given user imput.",\n                },\n            ]\n        }\n    ]\n)\n\

In [7]:
from openai import OpenAI
import zipfile
import os

client = OpenAI()

# Extract zip 
zip_path = "./HelpHerInvest/Data/final_dataset_20260224v2.csv.zip"
extract_dir = "./HelpHerInvest/Data/"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

csv2_path = os.path.join(extract_dir, "final_dataset_20260224v2.csv")

# Upload both files
file1 = client.files.create(
    file=open("./HelpHerInvest/Data/nlp_clean_stock_symbols.csv", "rb"),
    purpose="user_data"
)

file2 = client.files.create(
    file=open(csv2_path, "rb"),
    purpose="user_data"
)

response = client.responses.create(
    model="gpt-5-mini",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_file", "file_id": file1.id},
                {"type": "input_file", "file_id": file2.id},
                {
                    "type": "input_text",
                    "text": (
                        "Role: you are a data scientist specializing in stock market analysis. "
                        "Your task is to provide recommendations of stocks based on the user input.\n"
                        "User input: I like the fashion industry.\n"
                        "Instructions:\n"
                        "1. Analyze the user input.\n"
                        "2. Use file1 only, return a list of 20 relevant stock symbols.\n"
                        "3. Use file 2 only, rank the 20 relevant stock symbols you selected, the rank should reflect the highest potential for growth or profitability.\n"
                        "4. In your output, provide a brief explanation of your reasoning for selecting these stocks, and provide a justification for the ranking."
                    )
                },
            ]
        }
    ]
)

print(response.output_text)

Summary of approach
- Selection (which stocks to consider): I used file1 (company descriptions) only to identify 20 stocks that are clearly in or closely tied to the fashion/apparel/footwear/accessories/department-store/retail ecosystem (apparel brands, footwear, fashion retailers, off‑price retailers, department stores, and adjacent specialty retailers).
- Ranking (ordering these 20): I used file2 only (the 2010-02-28 snapshot) to rank the 20 symbols by forward potential. The ranking is driven primarily by the file2 forward-excess column (fwd_excess), with fwd_return as a supporting metric. Higher fwd_excess → higher expected outperformance vs. benchmark in this dataset, so higher rank.

Selected 20 fashion-relevant stocks (chosen from file1)
- LULU (Lululemon)
- NKE (Nike)
- VFC (VF Corp / brands like The North Face, Vans, etc.)
- RL (Ralph Lauren)
- TPR (Tapestry — Coach, Kate Spade, Stuart Weitzman)
- ANF (Abercrombie & Fitch)
- URBN (Urban Outfitters)
- GAP (Gap Inc.)
- TJX (TJX C